# Full Research Pipeline: Dataset → Training → Large-Scale Evaluation

This notebook executes the **complete experimental pipeline**:

1. **Generate synthetic transit networks** (25-node random graphs, Mumford-style)
2. **Train the inductive route generator** using PPO
3. **Run full benchmark evaluation** on Mandl + Mumford instances:
   - LC (pure neural)
   - Neural BCO (neural + optimization)
   - Classic BCO (heuristic only)

---

## 1. Global Setup & Paths

In [ ]:
import torch
import pickle
import csv
from pathlib import Path
from typing import List, Any
from tqdm import tqdm
from torch_geometric.loader import DataLoader

from hydra import initialize_config_dir, compose

from transport.routes_generator.citygraph_dataset import DynamicCityGraphDataset, get_dataset_from_config, MIXED
from transport.routes_generator.eval_route_generator import eval_model
from transport.routes_generator.torch_utils import dump_routes
from transport.routes_generator import inductive_route_learning, bee_colony
import transport.routes_generator.utils as lrnu

ROOT_DIR = Path.cwd().parent.parent
CFG_DIR = ROOT_DIR / "transport/routes_generator/cfg"
DATASETS_DIR = ROOT_DIR / "datasets"
MODEL_WEIGHTS_DIR = ROOT_DIR / "examples/data/model_weights"
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

TRAINED_MODEL_PATH = MODEL_WEIGHTS_DIR / "inductive_random_graphs_weighted_connectivity.pt"

RESULTS_DIR = Path("experiment_results")
RESULTS_DIR.mkdir(exist_ok=True)
UNSERVED_DIR = RESULTS_DIR / "unserved_demand_matrices"
UNSERVED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ROUTES_DIR = ROOT_DIR / "examples/route_generator/output_routes"

print(f"Config dir:       {CFG_DIR}")
print(f"Dataset will be saved to: {DATASETS_DIR}")
print(f"Trained model → {TRAINED_MODEL_PATH}")
print(f"Evaluation results → {RESULTS_DIR.resolve()}")
print(f"Output routes → {OUTPUT_ROUTES_DIR.resolve()}")

## 2. Step 1: Generate Synthetic Training Dataset

In [ ]:
dataset = DynamicCityGraphDataset(
    min_nodes=20,
    max_nodes=20,
    edge_keep_prob=0.7,
    data_type=MIXED,
    directed=False,
    fully_connected_demand=True,
    mumford_style=True,
    pos_only=False,
)

n_graphs = 1000
save_dir = DATASETS_DIR / "20_nodes_mixed"
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Generating {n_graphs} synthetic graphs (20 nodes, mixed demand)...")
graphs = [dataset.generate_graph(draw=False) for _ in tqdm(range(n_graphs), desc="Graphs")]

save_path = save_dir / "raw_graphs_1000.pkl"
with open(save_path, "wb") as f:
    pickle.dump(graphs, f)

print(f"Dataset saved to: {save_path}")

## 3. Step 2: Train the Inductive Route Generator (PPO)

In [ ]:
print("Starting model training...")

with initialize_config_dir(config_dir=str(CFG_DIR)):
    cfg_train = compose(
        config_name="ppo_20nodes.yaml",
        overrides=[
            "+run_name=inductive_training_20nodes",
            "++experiment.logdir=./logs_training",
        ]
    )

inductive_route_learning.setup_and_train(cfg_train)

## 4. Step 3: Large-Scale Evaluation Setup

In [ ]:
DATASETS = ["mandl", "mumford0", "mumford1", "mumford2", "mumford3"]

WEIGHT_STEPS = [round(i * 0.1, 1) for i in range(11)]
COMBINATIONS = [(dt, round(1.0 - dt, 1), 0.0) for dt in WEIGHT_STEPS]
EXPERIMENTS = [(ds, dt, rt, ct) for ds in DATASETS for dt, rt, ct in COMBINATIONS]

print(f"Evaluation plan: {len(EXPERIMENTS)} configurations ({len(DATASETS)} datasets × {len(COMBINATIONS)} weight pairs)")

## 5. CSV & Storage Utilities

In [ ]:
CSV_HEADER = [
    "dataset", "dt_weight", "rt_weight", "ct_weight",
    "ATT", "RTT", "median_conn", "median_conn_w", "cost", "d_un%", "d0", "d1", "d2"
]

METRIC_KEYS = [
    'ATT', 'RTT', 'median_connectivity', 'median_connectivity_weighted',
    'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$'
]

def init_csv(path: Path):
    if not path.exists():
        with open(path, "w", newline="") as f:
            csv.writer(f).writerow(CSV_HEADER)

def append_csv(path: Path, row: List[Any]):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow(row)

def save_unserved(tensor: torch.Tensor, method: str, tag: str):
    path = UNSERVED_DIR / f"{method}_{tag}_unserved.pt"
    torch.save(tensor.cpu(), path)

lc_csv    = RESULTS_DIR / "01_lc_results.csv"
neuro_csv = RESULTS_DIR / "02_neural_bco_results.csv"
bco_csv   = RESULTS_DIR / "03_classic_bco_results.csv"

for p in (lc_csv, neuro_csv, bco_csv):
    init_csv(p)

## 6. Core Silent Evaluation Functions

In [ ]:
def run_lc(cfg):
    DEVICE, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(cfg, "nn_construction_", weights_required=True)
    test_ds = get_dataset_from_config(cfg.eval.dataset)
    test_dl = DataLoader(test_ds, batch_size=cfg.batch_size)

    _, unserved, metrics, routes = eval_model(
        model, test_dl, cfg.eval, cost_obj,
        n_samples=cfg.get("n_samples", 1),
        return_routes=True,
        silent=True,
        device=DEVICE
    )
    dump_routes(run_name, routes.cpu())
    return metrics, unserved, routes

def run_bco(cfg, init_routes=None, neural=True):
    prefix = "neural_bco_" if neural else "bco_"
    DEVICE, run_name, writer, cost_obj, bee_model = lrnu.process_standard_experiment_cfg(cfg, prefix, weights_required=neural)
    test_ds = get_dataset_from_config(cfg.eval.dataset)
    test_dl = DataLoader(test_ds, batch_size=cfg.batch_size)

    if neural and bee_model is not None:
        bee_model.eval()

    out = lrnu.test_method(
        bee_colony, test_dl, cfg.eval, cfg.init, cost_obj,
        sum_writer=writer, silent=True,
        n_bees=cfg.n_bees, n_iterations=cfg.n_iterations,
        device=DEVICE, bee_model=bee_model if neural else None,
        return_routes=True, routes_tensor=init_routes
    )
    return out[-2], out[-3], out[-1]

## 7. Main Experiment Loop

In [ ]:
ROOT_DIR = Path.cwd().parent.parent
CFG_DIR = ROOT_DIR / "transport" / "routes_generator" / "cfg"
TRAINED_MODEL_PATH = ROOT_DIR / "examples" / "data" / "model_weights" / "inductive_random_graphs_weighted_connectivity.pt"
OUTPUT_ROUTES_DIR = ROOT_DIR / "examples/route_generator/output_routes"

for dataset, dt, rt, ct in tqdm(EXPERIMENTS, desc="Full Evaluation Sweep"):
    tag = f"{dataset.upper()}_DT{dt:.1f}"
    param = f"dt{dt:.1f}_rt{rt:.1f}_ct{ct:.1f}"

    base_ov = [
        f"+eval={dataset}",
        f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
        f"++experiment.cost_function.kwargs.route_time_weight={rt}",
        f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
    ]

    try:
        with initialize_config_dir(config_dir=str(CFG_DIR), version_base="1.3"):
            cfg_lc = compose("eval_model_mumford", overrides=base_ov + [
                f"+model.weights={TRAINED_MODEL_PATH}",
                f"++run_name=LC_{param}"
            ])
        lc_metrics, lc_unserved, _ = run_lc(cfg_lc)
        save_unserved(lc_unserved, "LC", tag)
        append_csv(lc_csv, [dataset, dt, rt, ct] + [
            round(lc_metrics.get(k, torch.tensor(0.0)).item(), 4)
            for k in METRIC_KEYS
        ])
        print(f" Success LC → {tag} | cost: {lc_metrics['cost'].item():.4f}")
    except Exception as e:
        print(f" Failed LC → {tag} | {e}")
        continue

    routes_filename = f"nn_construction_LC_{param}_routes.pkl"
    init_routes_path = OUTPUT_ROUTES_DIR / routes_filename

    if not init_routes_path.exists():
        print(f" Warning: Routes file not found: {init_routes_path}")
        continue

    try:
        with initialize_config_dir(config_dir=str(CFG_DIR), version_base="1.3"):
            cfg_n = compose("neural_bco_mumford", overrides=base_ov + [
                f"+model.weights={TRAINED_MODEL_PATH}",
                f"++run_name=NEURO_{param}",
                f"init.path={init_routes_path}"
            ])
        n_metrics, n_unserved, _ = run_bco(cfg_n, init_routes=None, neural=True)
        save_unserved(n_unserved, "NEURO", tag)
        append_csv(neuro_csv, [dataset, dt, rt, ct] + [
            round(n_metrics.get(k, torch.tensor(0.0)).item(), 4)
            for k in METRIC_KEYS
        ])
        print(f" Success Neural BCO → {tag} | cost: {n_metrics['cost'].item():.4f}")
    except Exception as e:
        print(f" Failed Neural BCO → {tag} | {e}")

    try:
        with initialize_config_dir(config_dir=str(CFG_DIR), version_base="1.3"):
            cfg_c = compose("bco_mumford", overrides=base_ov + [
                f"++run_name=BCO_{param}",
                f"init.path={init_routes_path}"
            ])
        c_metrics, c_unserved, _ = run_bco(cfg_c, init_routes=None, neural=False)
        save_unserved(c_unserved, "BCO", tag)
        append_csv(bco_csv, [dataset, dt, rt, ct] + [
            round(c_metrics.get(k, torch.tensor(0.0)).item(), 4)
            for k in METRIC_KEYS
        ])
        print(f" Success Classic BCO → {tag} | cost: {c_metrics['cost'].item():.4f}")
    except Exception as e:
        print(f" Failed Classic BCO → {tag} | {e}")

    print("-" * 60)

## 8. Final Report

In [ ]:
print("\n" + "=" * 80)
print("FULL RESEARCH PIPELINE SUCCESSFULLY COMPLETED".center(80))
print("=" * 80)
print(f"Synthetic dataset:     {save_path}")
print(f"Trained model:         {TRAINED_MODEL_PATH} ({TRAINED_MODEL_PATH.stat().st_size // 1024 // 1024} MB)")
print(f"Total experiments:     {len(EXPERIMENTS)}")
print("\nResults:")
print(f"  • LC           → {lc_csv}")
print(f"  • Neural BCO   → {neuro_csv}")
print(f"  • Classic BCO  → {bco_csv}")
print(f"  • Unserved     → {UNSERVED_DIR}/")
print(f"  • Routes       → output_routes/")
print("=" * 80)
print("You can now run analysis/plotting notebooks!")